# 7.7 · 特征选择 / Feature Selection

> **课程定位 / Where this fits**
> 第 7 课，**Part 7 · 模型评估与优化**。
> Lesson 7, **Part 7 · Model Evaluation & Tuning**.
>
> 不是特征越多越好。无关/冗余特征会拖慢训练、伤害可解释性、加重过拟合（尤其高维小样本）。**特征选择**从一堆特征里挑出真正有用的子集。三大流派：**filter（过滤式）、wrapper（包裹式）、embedded（嵌入式）**——这是面试的标准框架。
> More features isn't better. Irrelevant/redundant features slow training, hurt interpretability, and worsen overfitting (especially high-dim small-sample). **Feature selection** picks the genuinely useful subset. Three families: **filter, wrapper, embedded** — the standard interview framework.
>
> 💼 **实战/面试视角**："怎么做特征选择 / 三类方法区别 / RFE 是什么" 是建模流程常考。
> 💼 **Practical/interview angle:** "how to select features / the three families / what is RFE" — modeling-pipeline questions.

> 💡 **面试相关 / Interview-relevant**
> - "filter/wrapper/embedded 三类的区别与取舍"（出镜率 ★★★★★）
> - "为什么要特征选择"（★★★★）
> - "RFE 怎么工作"（★★★★）
> - "Lasso 怎么做特征选择"（★★★★，接 4.5）
> - "特征选择要不要放进 CV（防泄漏）"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解为什么要特征选择。
   Understand why feature selection matters.
2. 掌握 **filter**（方差/相关/互信息/ANOVA F）。
   Master filter methods (variance/correlation/mutual info/ANOVA F).
3. 掌握 **wrapper**（RFE）。
   Master wrapper methods (RFE).
4. 掌握 **embedded**（Lasso / 树重要性）。
   Master embedded methods (Lasso / tree importance).
5. 牢记特征选择必须在 **CV 内**（防泄漏）。
   Remember selection must live inside CV (no leakage).

## 目录 / TOC
1. [先建直觉 + 三大流派 ⭐](#1)
2. [📊 数据：真信号 + 大量噪声](#2)
3. [Filter：单变量过滤 ⭐](#3)
4. [Wrapper：RFE ⭐](#4)
5. [Embedded：Lasso / 树 ⭐](#5)
6. [防泄漏 + 对比 + 小结 ⭐](#6)


<a id="1"></a>
## 1. 先建直觉 + 三大流派 ⭐ / Intuition & the Three Families

为什么少即是多：**无关特征是噪声**（模型可能误把噪声当信号 → 过拟合）；**冗余特征**浪费容量、伤害可解释性；特征越少**训练越快、上线越简单**。
Why less is more: **irrelevant features are noise** (the model may mistake noise for signal → overfit); **redundant features** waste capacity and hurt interpretability; fewer features mean **faster training and simpler deployment**.

三大流派（核心框架）：
The three families (the core framework):
- **Filter（过滤式）**：**不用模型**，靠统计量（方差、相关、互信息、ANOVA F）给每个特征打分排序，留高分的。最快、最通用，但忽略特征间交互。
  **Filter:** **model-free**, scores each feature by a statistic (variance, correlation, mutual info, ANOVA F) and keeps the top ones. Fastest, most general, but ignores feature interactions.
- **Wrapper（包裹式）**：**反复训模型**评估不同特征子集（如 RFE 逐个剔除最弱的）。最准（直接优化模型表现），但最慢。
  **Wrapper:** **repeatedly trains a model** to evaluate feature subsets (e.g. RFE drops the weakest iteratively). Most accurate (directly optimizes model performance), but slowest.
- **Embedded（嵌入式）**：选择**内嵌在训练里**（Lasso 把系数压 0、树给重要性）。又快又好，是实战首选。
  **Embedded:** selection is **built into training** (Lasso zeros coefficients, trees give importances). Fast and good — the practical default.


<a id="2"></a>
## 2. 数据：真信号 + 大量噪声 / Data: Signal + Lots of Noise

用 `make_classification` 造一份"**Madelon 风格**"数据：20 个特征里只有 5 个真正有信息(informative)、5 个是冗余(redundant，由信息特征线性组合而成)、其余 10 个是纯噪声。目标是让各方法**找出那 5 个真信号特征**。
We build a "**Madelon-style**" dataset with `make_classification`: of 20 features, only 5 are truly informative, 5 are redundant (linear combos of the informative ones), and 10 are pure noise. The goal is for each method to **recover those 5 informative features**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")

# n_informative=5 真信号, n_redundant=5 冗余, 其余10个噪声 / 5 signal, 5 redundant, 10 noise
X, y = make_classification(n_samples=2000, n_features=20, n_informative=5, n_redundant=5,
                           n_repeated=0, shuffle=False, random_state=0)
# shuffle=False → 前5列是信息特征, 接着5列冗余, 后10列噪声 / known layout
feat_names = [f"信息{i}" for i in range(5)] + [f"冗余{i}" for i in range(5)] + [f"噪声{i}" for i in range(10)]
Xs = StandardScaler().fit_transform(X)
print(f"数据 {X.shape}: 5 信息 + 5 冗余 + 10 噪声特征; 看各方法能否找出真信号")


<a id="3"></a>
## 3. Filter：单变量过滤 ⭐ / Filter Methods

Filter 给每个特征单独打分。两个常用统计量：
Filter scores each feature independently. Two common statistics:
- **ANOVA F 检验**（`f_classif`）：衡量特征在不同类别间的**均值差异**（线性关系）。
  **ANOVA F-test:** measures the **difference in feature means across classes** (linear relationship).
- **互信息**（`mutual_info_classif`）：衡量特征和目标的**任意（含非线性）依赖**——比 F 检验更通用。
  **Mutual information:** measures **any (incl. nonlinear) dependence** — more general than the F-test.

Filter 的优点是快且与模型无关；缺点是**逐个看特征、看不到交互**（一个单独无用但和别人组合才有用的特征会被漏掉）。
Filter is fast and model-agnostic; its weakness is **judging features one at a time, missing interactions** (a feature useless alone but useful in combination gets dropped).


In [ ]:
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest

f_scores = f_classif(Xs, y)[0]                          # ANOVA F 分数(线性关系)
mi_scores = mutual_info_classif(Xs, y, random_state=0) # 互信息(任意依赖)
df_scores = pd.DataFrame({"feature": feat_names, "F": f_scores, "MI": mi_scores})
print("各特征得分(F 检验 / 互信息), 按 F 排序:")
print(df_scores.sort_values("F", ascending=False).head(8).round(3).to_string(index=False))

# SelectKBest 选 F 分数最高的 5 个 / keep top-5 by F
selected = np.array(feat_names)[SelectKBest(f_classif, k=5).fit(Xs, y).get_support()]
print(f"\nSelectKBest(F, k=5) 选中: {list(selected)}")
print("→ 信息/冗余特征得分高(都和目标相关), 噪声得分低; filter 能筛掉噪声但分不清信息vs冗余")


<a id="4"></a>
## 4. Wrapper：RFE ⭐ / Wrapper: RFE

**RFE（递归特征消除）** 是最常用的 wrapper：训一个模型 → 看哪个特征**最不重要** → 剔掉它 → 用剩下的重训 → 重复，直到剩下指定个数。因为它**直接用模型表现**来评判、且考虑特征组合，通常比 filter 更准；代价是要反复训练，慢。配合 `RFECV` 还能**自动选最优特征数**。
**RFE (Recursive Feature Elimination)** is the most common wrapper: train a model → find the **least important** feature → drop it → retrain on the rest → repeat until the target count remains. Because it **uses model performance** and considers combinations, it's usually more accurate than filter; the cost is repeated training (slow). `RFECV` even **auto-selects the optimal number of features**.


In [ ]:
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression

# RFE: 用逻辑回归反复剔除最弱特征, 直到剩 5 个 / recursively eliminate to 5
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=5).fit(Xs, y)
print(f"RFE(选5个) 选中: {list(np.array(feat_names)[rfe.support_])}")

# RFECV: 用 CV 自动决定该留几个特征 / auto-pick the number of features
rfecv = RFECV(LogisticRegression(max_iter=1000), cv=StratifiedKFold(5), scoring="accuracy").fit(Xs, y)
print(f"RFECV 自动选出 {rfecv.n_features_} 个特征: {list(np.array(feat_names)[rfecv.support_])}")
print("RFE 直接用模型表现评判 + 考虑特征组合 → 通常比 filter 准, 但反复训练较慢")


<a id="5"></a>
## 5. Embedded：Lasso / 树 ⭐ / Embedded: Lasso / Trees

Embedded 把特征选择**嵌进模型训练**，一步到位、又快又好：
Embedded folds selection **into training**, in one shot, fast and effective:
- **Lasso（L1 正则，4.5）**：训练时直接把无用特征的系数**压到 0**——非零系数对应的就是被选中的特征。
  **Lasso (L1, 4.5):** training drives useless coefficients to **exactly 0** — the nonzero ones are the selected features.
- **树/森林的特征重要性**：训练后用不纯度下降给每个特征打分（注意它对高基数特征有偏，更可靠用置换重要性，5.7）。
  **Tree/forest importances:** score features by impurity decrease after training (biased toward high-cardinality features; permutation importance is more reliable, 5.7).

`SelectFromModel` 能把任意带 `coef_`/`feature_importances_` 的模型变成特征选择器。
`SelectFromModel` turns any model with `coef_`/`feature_importances_` into a selector.


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier

# Lasso: 非零系数 = 被选中的特征 / nonzero Lasso coefficients = selected
lasso = LassoCV(cv=5, random_state=0, max_iter=10000).fit(Xs, y)
lasso_sel = np.array(feat_names)[np.abs(lasso.coef_) > 1e-6]
print(f"Lasso 选中(非零系数): {list(lasso_sel)}")

# 随机森林重要性 → SelectFromModel 自动按阈值选 / RF importance selection
sfm = SelectFromModel(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xs, y)
rf_sel = np.array(feat_names)[sfm.get_support()]
print(f"随机森林重要性选中: {list(rf_sel)}")
print("Embedded 一步到位(训练即选择), 又快又好 → 实战首选; Lasso 偏稀疏, 树重要性偏全")


<a id="6"></a>
## 6. 防泄漏 + 对比 + 小结 ⭐ / Leakage, Comparison & Summary

**最重要的纪律**（接 3.9）：特征选择**会看 y**，所以必须放进 **Pipeline**、在每折训练部分内做。如果先在全数据上选特征再 CV，就把验证折的标签泄漏进了选择过程——评估虚高（3.9 用纯噪声+随机标签证明过：错误做法能从噪声里"选出"虚假的高分）。
**The key discipline** (continuing 3.9): feature selection **looks at y**, so it must go inside a **Pipeline**, applied to each fold's training part. Selecting on all data before CV leaks validation-fold labels into selection — inflating the estimate (3.9 proved this with pure noise + random labels: the wrong way "selects" a fake high score from noise).


In [ ]:
from sklearn.pipeline import Pipeline

# ✅ 正确: 特征选择放进 Pipeline, CV 时只在每折 train 上选 / selection inside the pipeline
pipe = Pipeline([("select", SelectKBest(f_classif, k=5)),
                 ("clf", LogisticRegression(max_iter=1000))])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
acc_sel = cross_val_score(pipe, Xs, y, cv=cv).mean()
acc_all = cross_val_score(LogisticRegression(max_iter=1000), Xs, y, cv=cv).mean()
print(f"用全部 20 特征:        CV 准确率 = {acc_all:.4f}")
print(f"选 5 个特征(Pipeline内): CV 准确率 = {acc_sel:.4f}")
print("→ 选 5 个达到/接近全特征效果, 但模型更简单、更快、更可解释")
print("\n💡 防泄漏: 特征选择会看 y → 必须放进 Pipeline 在每折训练部分做(3.9/3.12)")


```
为什么选特征: 无关特征=噪声(过拟合), 冗余浪费容量, 少特征=更快+更可解释
三大流派:
  filter(无模型, 统计量打分: F检验/互信息): 最快, 但看不到特征交互
  wrapper(反复训模型评估子集: RFE/RFECV): 最准, 但慢
  embedded(选择内嵌训练: Lasso 系数归0 / 树重要性): 又快又好, 实战首选
防泄漏 ⭐: 特征选择会看 y → 必须放进 Pipeline 在每折训练部分做(否则虚高, 3.9)
```

### 💡 面试速查 / Interview cheat-sheet
1. **filter(快/无模型/忽略交互) vs wrapper(准/慢/RFE) vs embedded(快好/Lasso·树)**。
   filter (fast/model-free/ignores interactions) vs wrapper (accurate/slow/RFE) vs embedded (fast-good/Lasso·trees).
2. **RFE**: 反复剔除最弱特征; RFECV 自动选特征数。
   RFE recursively drops the weakest; RFECV auto-picks the count.
3. **Lasso(L1) 把系数压0** = 自动特征选择(4.5)。
   Lasso (L1) zeros coefficients = automatic selection.
4. **互信息抓非线性依赖**, F 检验只抓线性。
   Mutual information catches nonlinear dependence; the F-test only linear.
5. **特征选择必须在 CV 内**(看 y, 否则泄漏), 用 Pipeline。
   Feature selection must be inside CV (it sees y, else leaks); use a Pipeline.

### 下一节 / Next
**7.8 模型解释**——选好特征、调好模型后, 怎么向他人(和监管)解释"模型为什么这样预测"? SHAP、LIME、置换重要性、PDP/ICE。
**7.8 Model Interpretability** — after selecting features and tuning, how do you explain "why the model predicts this" to others (and regulators)? SHAP, LIME, permutation importance, PDP/ICE.
